# Fleet Intelligence Platform — Full Pipeline

This notebook runs the complete pipeline:
1. Data Generation
2. Feature Engineering
3. Carbon Model Training
4. Safety Model Training
5. HR/Churn Model Training
6. Cross-Impact Analysis

**Run in Google Colab** or locally. All outputs are saved to `data/`, `models/`, and `results/` directories.

After running, push the repo to GitHub and deploy on Streamlit Community Cloud.

## Setup

In [ ]:
# Install dependencies (uncomment for Colab)
# !pip install pandas numpy scikit-learn xgboost lightgbm shap matplotlib plotly joblib

In [ ]:
import os
import sys

# If running from notebooks/ directory, add parent to path
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, '.')

os.makedirs('data', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('results', exist_ok=True)

print(f'Working directory: {os.getcwd()}')

## Step 1: Generate Data

In [ ]:
%run generate_data.py

In [ ]:
import pandas as pd

# Verify generated data
for fname in ['drivers.csv', 'trips.csv', 'events.csv', 'fuel.csv', 'activity.csv', 'incidents.csv']:
    df = pd.read_csv(f'data/{fname}')
    print(f'{fname:20s} {df.shape[0]:>10,} rows x {df.shape[1]:>3} cols')

## Step 2: Feature Engineering

In [ ]:
%run -m src.feature_engineering

In [ ]:
features = pd.read_csv('data/weekly_features.csv')
print(f'Feature matrix: {features.shape}')
features.describe().round(2)

## Step 3: Carbon Model

In [ ]:
%run -m src.carbon_model

In [ ]:
from IPython.display import Image, display
display(Image('results/carbon_shap_summary.png', width=600))
display(Image('results/carbon_by_behavior.png', width=600))

## Step 4: Safety Model

In [ ]:
%run -m src.safety_model

In [ ]:
display(Image('results/safety_score_distribution.png', width=600))
display(Image('results/incident_roc.png', width=600))
display(Image('results/safety_shap_summary.png', width=600))

## Step 5: HR / Churn Model

In [ ]:
%run -m src.hr_model

In [ ]:
display(Image('results/churn_shap_summary.png', width=600))
display(Image('results/burnout_scatter.png', width=600))
display(Image('results/survival_curves.png', width=600))
display(Image('results/training_impact.png', width=600))

## Step 6: Cross-Impact Analysis

In [ ]:
%run -m src.cross_impact

In [ ]:
display(Image('results/cross_correlation_heatmap.png', width=600))
display(Image('results/intervention_comparison.png', width=600))

In [ ]:
# Final check: list all output files
import os

for directory in ['data', 'models', 'results']:
    print(f'\n=== {directory}/ ===')
    for f in sorted(os.listdir(directory)):
        size = os.path.getsize(f'{directory}/{f}')
        if size > 1024*1024:
            print(f'  {f:40s} {size/1024/1024:.1f} MB')
        else:
            print(f'  {f:40s} {size/1024:.1f} KB')

## Done!

All artifacts are saved. To deploy:

1. Push this repo to GitHub
2. Go to [Streamlit Community Cloud](https://share.streamlit.io)
3. Deploy `app.py` from the repo
4. The app will install `requirements.txt` and serve the dashboard

**Note:** The raw data files (`trips.csv`, `events.csv`, `fuel.csv`, `activity.csv`) are in `.gitignore` since the Streamlit app only uses the aggregated `weekly_features.csv` and model outputs.